# TEMPLATE · Kaggle submission for a team model

**How to use:**
1. On Kaggle: **Create → Import Notebook** → upload this file. Rename it `casmi-<yourname>-<model>`.
2. Right panel → **Add Input**: the competition data, the dataset **casmi-team-src** (team code), and **your own weights dataset**.
3. Right panel → Settings: **Internet OFF**. Use a GPU only if your model needs one.
4. Replace the model cell below with your model class (copy it from your `members/<you>/model_*.py`).
5. **Save Version → Save & Run All**. When it finishes, open the version → **Submit to Competition**.

In [ ]:
import glob, os, sys, time
t0 = time.time()

# Team code (src/) from the casmi-team-src dataset
BASE = glob.glob("/kaggle/input/**/src/models/base.py", recursive=True)[0]
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(BASE))))

# RDKit is needed for fingerprints. If it's missing, install it from the team wheels dataset.
try:
    import rdkit
except ImportError:
    os.system("pip install --no-index --find-links /kaggle/input/casmi-wheels rdkit -q")

from src.data.build_reference import build
from src.evaluation.validate import make_submission
from src.models.base import Scorer, CosineScorer

TEST = glob.glob("/kaggle/input/**/test.parquet", recursive=True)[0]
DATA = os.path.dirname(TEST)
REF = "/kaggle/working/ref"
build(os.path.join(DATA, "train.parquet"), REF)   # ~3-5 min
print(f"reference built in {time.time() - t0:.0f}s")

## Your model
Replace `CosineScorer()` with your model. Weights are under `/kaggle/input/<your-weights-dataset>/`.

In [ ]:
# Example:
# WEIGHTS = glob.glob("/kaggle/input/**/model.pt", recursive=True)[0]
# class MyModel(Scorer):
#     name = "fingerprint-mlp"
#     def __init__(self, path): ...
#     def score(self, spectra, candidates): ...
# model = MyModel(WEIGHTS)

model = CosineScorer()

In [ ]:
sub = make_submission(model, TEST, REF, "/kaggle/working/submission.csv")
for f in os.listdir(REF):          # keep only submission.csv in the output
    os.remove(os.path.join(REF, f))
os.rmdir(REF)
print(sub.shape, f"total {time.time() - t0:.0f}s")
sub.head()